In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import warnings

warnings.filterwarnings("ignore")

In [2]:
FS = 700                 # Sampling rate (Hz)
WINDOW_DURATION = 60     # seconds
WINDOW_SIZE = FS * WINDOW_DURATION
VALID_LABELS = [1, 2, 3]  # Baseline, Stress, Relaxed

In [3]:
def create_windows(signal, labels, window_size):
    windows = []
    window_labels = []

    for start in range(0, len(signal) - window_size, window_size):
        end = start + window_size
        label_segment = labels[start:end]

        # Remove undefined labels (0)
        label_segment = label_segment[label_segment != 0]
        if len(label_segment) == 0:
            continue

        label = np.bincount(label_segment).argmax()
        windows.append(signal[start:end])
        window_labels.append(label)

    return windows, window_labels


In [4]:
def extract_hrv_features(ecg_signal, fs=FS):
    # Normalize ECG
    ecg_signal = (ecg_signal - np.mean(ecg_signal)) / np.std(ecg_signal)

    # Detect R-peaks
    peaks, _ = find_peaks(ecg_signal, distance=0.6 * fs)

    if len(peaks) < 10:
        return None

    # RR intervals (seconds)
    rr_intervals = np.diff(peaks) / fs
    if len(rr_intervals) < 5:
        return None

    mean_rr = np.mean(rr_intervals)
    sdnn = np.std(rr_intervals)
    rmssd = np.sqrt(np.mean(np.diff(rr_intervals) ** 2))
    mean_hr = 60 / mean_rr

    return {
        "Mean_RR": mean_rr,
        "SDNN": sdnn,
        "RMSSD": rmssd,
        "Mean_HR": mean_hr
    }


In [5]:
def extract_resp_features(resp_signal, fs=FS):
    resp_signal = resp_signal - np.mean(resp_signal)

    # Zero-crossing based breathing rate
    zero_crossings = np.where(np.diff(np.sign(resp_signal)))[0]
    duration_sec = len(resp_signal) / fs

    if duration_sec == 0:
        return None

    breaths = len(zero_crossings) / 2
    breath_rate = (breaths / duration_sec) * 60
    resp_variability = np.std(resp_signal)

    return {
        "Resp_Rate": breath_rate,
        "Resp_Variability": resp_variability
    }


In [6]:
DATA_DIR = "../data/raw/WESAD"
subjects = sorted([s for s in os.listdir(DATA_DIR) if s.startswith("S")])

feature_rows = []

for subject in subjects:
    file_path = os.path.join(DATA_DIR, subject, f"{subject}.pkl")

    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    chest = data["signal"]["chest"]
    ecg = chest["ECG"].flatten()
    resp = chest["Resp"].flatten()
    labels = data["label"].flatten()

    ecg_windows, window_labels = create_windows(ecg, labels, WINDOW_SIZE)
    resp_windows, _ = create_windows(resp, labels, WINDOW_SIZE)

    print(f"{subject}: windows = {len(ecg_windows)}")

    for ecg_w, resp_w, label in zip(ecg_windows, resp_windows, window_labels):
        if label not in VALID_LABELS:
            continue

        ecg_feats = extract_hrv_features(ecg_w)
        resp_feats = extract_resp_features(resp_w)

        if ecg_feats is None or resp_feats is None:
            continue

        row = {
            "Subject": subject,
            "Label": label
        }
        row.update(ecg_feats)
        row.update(resp_feats)

        feature_rows.append(row)

S10: windows = 62
S11: windows = 60
S13: windows = 60
S14: windows = 61
S15: windows = 61
S16: windows = 62
S17: windows = 62
S2: windows = 57
S3: windows = 61
S4: windows = 60
S5: windows = 60
S6: windows = 59
S7: windows = 60
S8: windows = 62
S9: windows = 59


In [7]:
df_features = pd.DataFrame(feature_rows)
df_features.shape

(599, 8)

In [8]:
print(df_features.head())
print(df_features.columns.tolist())


  Subject  Label   Mean_RR      SDNN     RMSSD    Mean_HR  Resp_Rate  \
0     S10      1  0.725557  0.144507  0.140624  82.695032       79.5   
1     S10      1  0.685781  0.128872  0.082190  87.491522      103.5   
2     S10      1  0.859689  0.261279  0.266298  69.792645      100.0   
3     S10      1  0.887569  0.233748  0.249909  67.600356      124.5   
4     S10      1  0.674756  0.147010  0.171729  88.920967       90.0   

   Resp_Variability  
0          3.347959  
1          3.421302  
2          1.961128  
3          2.406354  
4          2.351720  
['Subject', 'Label', 'Mean_RR', 'SDNN', 'RMSSD', 'Mean_HR', 'Resp_Rate', 'Resp_Variability']


In [ ]:
os.makedirs("../data/processed", exist_ok=True)
df_features.to_csv("../data/processed/windows_features.csv", index=False)